# Lab 3.1.3 — Create a machine learning model

**Learning objective:** select, train, and test a supervised classification model, then explain the difference between evaluation/tuning and testing by comparing validation and test accuracy.

This notebook follows **predict → act → observe → explain**. Keep `learning_log.md` open beside it. At every **STOP**, write before you run.

## Learning agreement

- Code is evidence, not the explanation.
- Use the first hint only after an attempt.
- Treat the test set as a sealed exam: do not inspect it, tune on it, or repeatedly score it.
- A score on 30 test examples is an estimate with sampling uncertainty, not a permanent fact about the algorithm.

The Iris dataset is deliberately small and clean. That isolates the model workflow for 3.1.3; section 3.2 will make data preparation the focus.

In [ ]:
# Preflight: this cell should run without edits.
import sys
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

RANDOM_SEED = 42
print(f"Python: {sys.version.split()[0]}")
print(f"scikit-learn: {sklearn.__version__}")

## 1. Translate the task into ML language

We will predict an iris species from four flower measurements.

**STOP — write in the log:**

1. What is one example?
2. Which values are features, and which value is the label?
3. Why is this supervised learning?
4. Why is it classification rather than ML regression?

<details><summary>Concept nudge</summary>Supervised learning has known answers during training. Classification predicts membership in a discrete set; regression predicts a continuous numerical value.</details>

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

print(iris.DESCR.split("Data Set Characteristics:")[1].split("References")[0])
print("Feature names:", list(iris.feature_names))
print("Class names:", list(iris.target_names))
print("X shape:", X.shape, "| y shape:", y.shape)

### Make a baseline expectation

Before building a model, ask what accuracy a trivial strategy could achieve. If a classifier always predicts the most frequent species, what accuracy would you expect here?

**STOP:** calculate from the dataset description before running the next cell. Why is beating this baseline necessary but not sufficient? Before seeing model results, propose a provisional success criterion for this lab. Then name at least one criterion a real stakeholder would need beyond overall accuracy.

In [ ]:
class_counts = np.bincount(y)
majority_baseline = class_counts.max() / class_counts.sum()
print("Class counts:", dict(zip(iris.target_names, class_counts)))
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

## 2. Build three subsets with three different jobs

Use **60% training, 20% validation, and 20% test**. We will split twice: first seal away test data, then take validation data from what remains.

**STOP — reason before editing:**

- After reserving 20%, what fraction remains for development?
- What fraction of that remainder must become validation so validation is 20% of the original?
- Why pass the labels to `stratify`?
- What does the seed control, and what does it *not* guarantee?

Replace the two `None` values below.

<details><summary>Hint 1 — arithmetic</summary>The second percentage is relative to the remainder, not the original dataset. Solve <code>remainder × second_fraction = 0.20</code>.</details>
<details><summary>Hint 2 — API</summary><code>train_test_split</code> can split matching feature and label arrays together. An integer <code>random_state</code> makes the shuffle reproducible; <code>stratify=y</code> approximately preserves class proportions.</details>

In [ ]:
TEST_FRACTION = None                 # TODO: fraction of the original data
VALIDATION_FRACTION_OF_REMAINDER = None  # TODO: fraction of the post-test remainder

assert TEST_FRACTION is not None, "Choose TEST_FRACTION after writing your reasoning."
assert VALIDATION_FRACTION_OF_REMAINDER is not None, "Choose the second fraction."

X_remainder, X_test, y_remainder, y_test = train_test_split(
    X, y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_SEED,
    stratify=y,
)

X_train, X_validation, y_train, y_validation = train_test_split(
    X_remainder, y_remainder,
    test_size=VALIDATION_FRACTION_OF_REMAINDER,
    random_state=RANDOM_SEED,
    stratify=y_remainder,
)

In [ ]:
# Audit the split. A useful workflow checks assumptions instead of trusting variable names.
expected_sizes = (90, 30, 30)
actual_sizes = (len(X_train), len(X_validation), len(X_test))
assert actual_sizes == expected_sizes, f"Expected {expected_sizes}, got {actual_sizes}"
assert sum(actual_sizes) == len(X)

for name, labels in [
    ("train", y_train),
    ("validation", y_validation),
    ("test (counts only; still sealed)", y_test),
]:
    counts = np.bincount(labels, minlength=len(iris.target_names))
    print(f"{name:34s} n={len(labels):3d} class counts={counts.tolist()}")

### Inspect development evidence, not test evidence

The plot below uses training rows only. Predict which two species will be easiest and hardest to separate. Then run it.

Why is a two-feature view useful for intuition but incomplete evidence about a model that receives four features?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for class_id, class_name in enumerate(iris.target_names):
    mask = y_train == class_id
    ax.scatter(
        X_train[mask, 2], X_train[mask, 3],
        label=class_name, alpha=0.8, edgecolor="black", linewidth=0.4,
    )
ax.set_xlabel(iris.feature_names[2])
ax.set_ylabel(iris.feature_names[3])
ax.set_title("Training data only")
ax.legend()
plt.show()

## 3. Select a classifier family, then train it

Our constraints are: four numeric features, three classes, a tiny dataset, no preprocessing in this exercise, an inspectable model, and one intuitive capacity control. A decision tree is a reasonable selection because it can express feature-threshold rules directly and does not require feature scaling. It is not the uniquely correct choice: nearest-neighbor and logistic classifiers, among others, could also solve this task with different assumptions and preprocessing needs.

**STOP:** defend or challenge the decision-tree selection. Name one alternative and one trade-off it introduces.

A decision tree learns a sequence of feature-threshold questions. `max_depth` limits how many questions can be chained along a path. It is a **hyperparameter**: we choose it; training learns the actual thresholds.

**STOP:** choose an initial depth from 1–4 before seeing any score. Predict training accuracy, validation accuracy, and their gap. Record your reasoning.

<details><summary>API nudge</summary><code>fit(X_train, y_train)</code> learns from labeled training examples. <code>predict(X_validation)</code> uses the learned tree without receiving the correct validation labels.</details>

In [ ]:
INITIAL_DEPTH = None  # TODO: choose an integer from 1 through 4
assert INITIAL_DEPTH in {1, 2, 3, 4}, "Choose an initial depth from 1 through 4."

first_model = DecisionTreeClassifier(
    max_depth=INITIAL_DEPTH,
    random_state=RANDOM_SEED,
)
first_model.fit(X_train, y_train)

first_train_predictions = first_model.predict(X_train)
first_validation_predictions = first_model.predict(X_validation)
first_train_accuracy = accuracy_score(y_train, first_train_predictions)
first_validation_accuracy = accuracy_score(y_validation, first_validation_predictions)

print(f"Training accuracy:   {first_train_accuracy:.3f}")
print(f"Validation accuracy: {first_validation_accuracy:.3f}")
print(f"Gap (train - validation): {first_train_accuracy - first_validation_accuracy:+.3f}")

### Explain before tuning

- If training accuracy is low, what might that say about the tree's capacity?
- If training accuracy is perfect but validation accuracy drops, what might that say about generalization?
- Can one 30-example validation set prove either diagnosis? What else could cause a gap?

Avoid the shortcut “higher training accuracy means a better model.” Training accuracy tells you how well the model fits evidence it was allowed to learn from. The validation result estimates performance on unseen-but-development-visible evidence.

## 4. Evaluate candidates and tune one hyperparameter

Now use the validation set to compare depths. Include a shallow model, several deeper models, and `None` for unlimited depth. Order them from simplest to most flexible so a tie naturally favors the simpler candidate.

**STOP:** sketch the training and validation trends you expect. Then replace the empty list.

<details><summary>Hint</summary>A suitable candidate list is <code>[1, 2, 3, 4, None]</code>. Here <code>None</code> means the tree can grow until another stopping rule applies.</details>

In [ ]:
candidate_depths = []  # TODO
assert len(candidate_depths) >= 4 and None in candidate_depths, (
    "Compare at least four depths, including None."
)

validation_results = []
for depth in candidate_depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED)
    model.fit(X_train, y_train)

    train_accuracy = accuracy_score(y_train, model.predict(X_train))
    validation_accuracy = accuracy_score(y_validation, model.predict(X_validation))
    validation_results.append({
        "depth": depth,
        "train_accuracy": train_accuracy,
        "validation_accuracy": validation_accuracy,
        "model": model,
    })
    print(
        f"depth={str(depth):>4s} | train={train_accuracy:.3f} "
        f"| validation={validation_accuracy:.3f} "
        f"| gap={train_accuracy - validation_accuracy:+.3f}"
    )

### Select using validation evidence

The next cell chooses the highest validation accuracy. Python's `max` returns the first tied item, so the simple-to-flexible ordering acts as an explicit simplicity tie-break.

Before running it, answer: is this choice independent of the validation data? Is it independent of the test data?

In [ ]:
selected_result = max(
    validation_results,
    key=lambda result: result["validation_accuracy"],
)
selected_model = selected_result["model"]
selected_validation_accuracy = selected_result["validation_accuracy"]

print("Selected max_depth:", selected_result["depth"])
print(f"Selected validation accuracy: {selected_validation_accuracy:.3f}")

## 5. Open the test vault once

The model and selection rule are now fixed. Before continuing, record: expected test accuracy, expected `test − validation` difference, and how you would interpret a large negative difference.

Set the confirmation below to `True` only after writing. Then run the test exactly once.

The test score may be above or below validation accuracy. Independence concerns the *role the data played*, not an ordering rule for two noisy estimates.

In [ ]:
I_RECORDED_MY_TEST_PREDICTION = False  # TODO: change only after writing in the log
assert I_RECORDED_MY_TEST_PREDICTION, "Write and lock your prediction first."

test_predictions = selected_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
validation_test_difference = test_accuracy - selected_validation_accuracy

print(f"Validation accuracy used for selection: {selected_validation_accuracy:.3f}")
print(f"Independent test accuracy:             {test_accuracy:.3f}")
print(f"Difference (test - validation):         {validation_test_difference:+.3f}")

### Reconstruct accuracy instead of treating it as magic

Write the formula for accuracy, calculate the number of correct predictions you expect from the displayed score and 30 test examples, then verify below.

In [ ]:
correct_predictions = np.sum(test_predictions == y_test)
manual_accuracy = correct_predictions / len(y_test)
assert np.isclose(manual_accuracy, test_accuracy)
print(f"{correct_predictions} correct / {len(y_test)} total = {manual_accuracy:.3f}")

## 6. Explain the difference the syllabus asks for

Complete the teach-back in `learning_log.md` before expanding this checkpoint.

<details><summary>Explanation checkpoint — compare after writing yours</summary>Training data directly determined the tree's learned thresholds. Validation data did not fit those thresholds, but its accuracy guided our choice of <code>max_depth</code>, so it participated in model development. Test data influenced neither learned parameters nor the hyperparameter choice; we used it once after selection to obtain a more independent estimate. Similar validation and test accuracy would increase confidence that selection did not exploit quirks of the validation sample, while a difference may reflect sampling variation, overfitting to development choices, or unrepresentative splits. Neither score proves operational fitness because Iris is small, unusually clean, and not tied here to real acceptance criteria or operational data.</details>

### Bug hunt

For each proposed action, name the dataset's *effective role* and decide whether the reported final score remains independent:

1. Try five depths, look at test accuracy for each, and report the best.
2. Choose depth on validation, test once, dislike the result, change depth, and test again.
3. Use validation to choose depth, then train a fresh model on training + validation before the one final test.
4. Repeat this entire notebook with many random seeds and publish the seed with the best test result.

Question 3 is a legitimate common workflow when specified in advance: validation contributes to development and the untouched test set remains the final check. We did not add that extra step here because this lab's central comparison is clearest when the same selected fitted model produces both scores.

## Completion check

You are finished when you can do all of these without reading code line by line:

- [ ] Point to the features and labels.
- [ ] Explain `fit` and `predict`.
- [ ] State the distinct jobs of train, validation, and test.
- [ ] Explain how validation accuracy changed a model-development decision.
- [ ] Report validation and test accuracy and interpret their difference cautiously.
- [ ] Explain why further tuning on this test result would invalidate its role.
- [ ] Name at least two reasons this toy result is not a production acceptance test.

For deeper reading, use `RESOURCES.md`. For retention, close the notebook and answer the five retrieval questions in `learning_log.md`.